In [1]:
import os
import random

import albumentations as A
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Keep deterministic behavior for repeatability (stability helps score move predictably).
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda


In [2]:
model_path = "../input/en-b4-tta-calr-40/model_40.pth"
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

train_csv_path = "../input/cassava-leaf-disease-classification/train.csv"
train_images_path = "../input/cassava-leaf-disease-classification/train_images"

assert os.path.exists(sample_sub_path), f"Missing sample submission: {sample_sub_path}"
assert os.path.isdir(test_images_path), f"Missing test_images dir: {test_images_path}"
assert os.path.exists(train_csv_path), f"Missing train.csv: {train_csv_path}"
assert os.path.isdir(
    train_images_path
), f"Missing train_images dir: {train_images_path}"

has_ckpt = os.path.exists(model_path)
print("Checkpoint found:", has_ckpt, "| path:", model_path)



Checkpoint found: False | path: ../input/en-b4-tta-calr-40/model_40.pth


In [3]:
# Minimal fix to move score toward target:
# If cassava checkpoint is missing, we must fine-tune the (same) EfficientNet-B4 on cassava
# because the 5-class classifier head is otherwise random -> near-random accuracy.
if has_ckpt:
    model = models.efficientnet_b4(weights=None)
else:
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)
model = model.to(device)


def _strip_prefix(sd, prefix):
    if (
        isinstance(sd, dict)
        and len(sd) > 0
        and all(k.startswith(prefix) for k in sd.keys())
    ):
        return {k[len(prefix) :]: v for k, v in sd.items()}
    return sd


if has_ckpt:
    ckpt = torch.load(model_path, map_location="cpu")
    state_dict = (
        ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    )
    state_dict = _strip_prefix(state_dict, "module.")
    state_dict = _strip_prefix(state_dict, "model.")
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(
        f"Loaded checkpoint. Missing keys: {len(missing)}, unexpected keys: {len(unexpected)}"
    )
else:
    print(
        "INFO: No cassava checkpoint found; will fine-tune ImageNet pretrained backbone on train.csv."
    )



Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 114MB/s]


INFO: No cassava checkpoint found; will fine-tune ImageNet pretrained backbone on train.csv.


In [4]:
# Albumentations pipelines:
# - Train aug is modest and label-preserving (keeps core approach; avoids heavy changes).
# - Val/Test uses the same normalization as inference to prevent distribution mismatch.
train_aug = A.Compose(
    [
        A.Resize(512, 512, p=1.0),
        A.RandomCrop(380, 380, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Transpose(p=0.2),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)

val_aug = A.Compose(
    [
        A.Resize(512, 512, p=1.0),
        A.CenterCrop(380, 380, p=1.0),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)

sub_aug = A.Compose(
    [
        A.Resize(512, 512, p=1.0),
        A.CenterCrop(380, 380, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Transpose(p=0.2),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)


def to_tensor_hwc(img_hwc: np.ndarray) -> torch.Tensor:
    if img_hwc.ndim != 3 or img_hwc.shape[2] != 3:
        raise ValueError(f"Expected HWC RGB image, got shape {img_hwc.shape}")
    return (
        torch.from_numpy(img_hwc.astype(np.float32, copy=False))
        .permute(2, 0, 1)
        .contiguous()
    )




In [5]:
from torch.utils.data import Dataset, DataLoader


class CassavaDataset(Dataset):
    def __init__(
        self, df: pd.DataFrame, images_dir: str, aug: A.Compose, has_labels: bool = True
    ):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.aug = aug
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["image_id"])
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)
        img_aug = self.aug(image=img_np)["image"]
        x = to_tensor_hwc(img_aug)
        if self.has_labels:
            y = int(row["label"])
            return x, torch.tensor(y, dtype=torch.long)
        return x, row["image_id"]




In [6]:
# Deterministic stratified split without adding new dependencies.
train_df = pd.read_csv(train_csv_path)
num_classes = 5

# Stratified split: per-class shuffle then allocate fixed fraction to validation.
val_frac = 0.1
rng = np.random.RandomState(SEED)

val_indices = []
for c in range(num_classes):
    idx = np.where(train_df["label"].values == c)[0]
    rng.shuffle(idx)
    n_val = max(1, int(round(len(idx) * val_frac)))
    val_indices.extend(idx[:n_val])

val_mask = np.zeros(len(train_df), dtype=bool)
val_mask[val_indices] = True

df_train = train_df.loc[~val_mask].reset_index(drop=True)
df_val = train_df.loc[val_mask].reset_index(drop=True)

print("Train/Val sizes:", len(df_train), len(df_val))
print("Train label dist:", df_train["label"].value_counts().sort_index().to_dict())
print("Val label dist:", df_val["label"].value_counts().sort_index().to_dict())




Train/Val sizes: 16849 1872
Train label dist: {0: 845, 1: 1711, 2: 1882, 3: 10371, 4: 2040}
Val label dist: {0: 94, 1: 190, 2: 209, 3: 1152, 4: 227}


In [7]:
# Fine-tuning only when missing checkpoint (minimal change to fix near-random head).
# Keep training simple and bounded to meet runtime constraints; no early stopping.
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


if not has_ckpt:
    batch_size = 32 if device.type == "cuda" else 16
    num_workers = 2

    train_loader = DataLoader(
        CassavaDataset(df_train, train_images_path, train_aug, has_labels=True),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )
    val_loader = DataLoader(
        CassavaDataset(df_val, train_images_path, val_aug, has_labels=True),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    criterion = nn.CrossEntropyLoss()

    # Small, stable optimizer setup for fine-tuning; preserves core model (EffNet-B4 + CE).
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

    epochs = 2  # minimal epochs to move score toward target without exceeding time constraints
    model.train()
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        running_acc = 0.0
        n_batches = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_acc += accuracy_from_logits(logits.detach(), y)
            n_batches += 1

        train_loss = running_loss / max(1, n_batches)
        train_acc = running_acc / max(1, n_batches)

        # Validation
        model.eval()
        v_loss = 0.0
        v_acc = 0.0
        v_batches = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logits = model(x)
                loss = criterion(logits, y)
                v_loss += loss.item()
                v_acc += accuracy_from_logits(logits, y)
                v_batches += 1
        val_loss = v_loss / max(1, v_batches)
        val_acc = v_acc / max(1, v_batches)

        print(
            f"Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )
        model.train()

    model.eval()
else:
    model.eval()



Epoch 1/2 | train_loss=0.5741 train_acc=0.7980 | val_loss=0.4121 val_acc=0.8607
Epoch 2/2 | train_loss=0.3820 train_acc=0.8681 | val_loss=0.3926 val_acc=0.8708


In [8]:
sample_sub = pd.read_csv(sample_sub_path)

predictions = []
tta_passes = 5  # keep original intent

with torch.no_grad():
    for _, sample_row in sample_sub.iterrows():
        img_path = os.path.join(test_images_path, sample_row.image_id)
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)

        logits_sum = None
        for _ in range(tta_passes):
            aug_img = sub_aug(image=img_np)["image"]  # float32 HWC normalized
            x = to_tensor_hwc(aug_img).unsqueeze(0).to(device, non_blocking=True)

            out = model(x)
            logits_sum = out if logits_sum is None else (logits_sum + out)

        logits_avg = logits_sum / float(tta_passes)
        pred_label = int(torch.argmax(logits_avg, dim=1).item())
        predictions.append([sample_row.image_id, pred_label])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
# Ensure alignment and exact required columns.
sub_df = sample_sub[["image_id"]].merge(sub_df, on="image_id", how="left")
sub_df["label"] = sub_df["label"].astype(int)

sub_df.to_csv("submission.csv", index=False)
print(sub_df.head())
print("Wrote submission.csv with shape:", sub_df.shape)
print("Label counts:", sub_df["label"].value_counts().sort_index().to_dict())

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with shape: (2676, 2)
Label counts: {0: 123, 1: 260, 2: 269, 3: 1694, 4: 330}
